## Rate Limits and Caching

**Rate limits:** API provider restrictions on how many requests or tokens you can process per minute.
**Caching:** storing results so you don't make the same API call twice.
Both directly affect cost, reliability, and user experience in production.

## How it actually works

**Rate limits:**
Every API has two limits:
- Requests per minute (RPM) — how many calls you can make
- Tokens per minute (TPM) — how many tokens across all calls

Exceed either limit — you get a 429 error. Your application breaks.
Claude API limits vary by tier. Higher spend = higher limits.

**Handling rate limits — five techniques:**
1. **Exponential backoff:** on 429 error, wait 1 second, retry. Fail again — wait 2 seconds. Then 4. Then 8. Never hammer a rate-limited API.
2. **Request queuing:** put requests in a queue, process at controlled rate within limits
3. **Response caching:** same question from 100 users — answer once, serve cached response to 99
4. **Prompt caching:** Claude charges fewer tokens for cached system prompts — reduces TPM consumption
5. **User-level throttling:** limit requests per user per minute so one heavy user doesn't consume the entire quota

**Two types of caching:**
- **Prompt caching:** Claude remembers your system prompt between calls. Charges fewer input tokens. Faster response.
- **Response caching:** your application stores the complete answer. Subsequent identical requests never reach Claude at all.

Prompt caching saves tokens.
Response caching saves money entirely.

## Where it breaks or gets dangerous

**Rate limit trap:**
500 users submit requests simultaneously at 9am Monday.
No queue, no throttling — all 500 hit the API at once.
Rate limit exceeded. Everyone gets errors. Application appears broken.
A queue would have served everyone with a slight delay instead.

**Cache invalidation:**
Cached responses go stale. Policies change, data updates, configurations change.
A cached answer about D365 approval thresholds from 6 months ago may be wrong today.
Always set cache expiry. Never cache indefinitely.

**Prompt caching gotcha:**
Prompt caching only works when the cached portion is identical across calls.
If your system prompt changes every request — caching does nothing.
Design system prompts to be stable. Put dynamic content in the user message, not the system prompt.

**Response caching privacy risk:**
Cached responses may contain data from one user served to another.
Never cache responses containing personal data, financial details, or user-specific information.
Cache only generic, non-personalised responses.

In [ ]:
# Rate Limits and Caching — conceptual illustration

import time
import hashlib
import json

# Simple in-memory response cache
response_cache = {}

def get_cache_key(prompt, model):
    """Generate a cache key from prompt and model."""
    content = f"{model}:{prompt}"
    return hashlib.md5(content.encode()).hexdigest()

def cached_api_call(prompt, model="claude-haiku-4-5"):
    """
    Simulates a cached API call.
    Real implementation would call the actual API.
    """
    cache_key = get_cache_key(prompt, model)
    
    if cache_key in response_cache:
        print(f"CACHE HIT — returning stored response (no API call made)")
        return response_cache[cache_key], "cached"
    
    # Simulate API call
    print(f"CACHE MISS — calling API...")
    time.sleep(0.1)  # simulate latency
    response = f"This is the answer to: {prompt}"
    
    # Store in cache
    response_cache[cache_key] = response
    print(f"Response stored in cache")
    return response, "api_call"

def exponential_backoff_demo():
    """Demonstrates exponential backoff strategy."""
    max_retries = 5
    wait_time = 1
    
    print("Exponential backoff simulation:")
    for attempt in range(max_retries):
        print(f"  Attempt {attempt + 1}: ", end="")
        # Simulate rate limit on first 3 attempts
        if attempt < 3:
            print(f"429 Rate Limited — waiting {wait_time}s before retry")
            wait_time *= 2
        else:
            print("Success!")
            break

# Test caching
print("=== RESPONSE CACHING ===")
q = "What is three-way matching in D365?"

response1, source1 = cached_api_call(q)
print(f"First call source: {source1}")
print()

response2, source2 = cached_api_call(q)
print(f"Second call source: {source2}")
print(f"Same response: {response1 == response2}")
print()

print("=== EXPONENTIAL BACKOFF ===")
exponential_backoff_demo()
print()

print("Cache stats:")
print(f"Total cached responses: {len(response_cache)}")
print(f"API calls saved on repeated queries: priceless")

## My D365 analogy

Rate limits in AI APIs work exactly like connection pool limits in D365.

Too many simultaneous AOS connections — the system throttles new requests.
You configure connection pooling to manage this — controlled throughput,
queued requests, graceful degradation instead of hard failures.

Response caching is like D365 static reference data caching.
Chart of accounts, financial dimension values, posting profiles —
you don't query these from the database on every transaction.
You load them once, cache them in memory, refresh on a schedule.

Same principle. Same risk — stale cache causing wrong answers.
Same solution — cache expiry and deliberate invalidation.

The engineer who has managed D365 performance tuning
already understands every concept in this notebook.
The vocabulary is different. The problems are identical.

## LinkedIn post idea

500 D365 users logging in simultaneously after a long weekend
is a connection pool problem.
500 users asking the same AI assistant the same question simultaneously
is a rate limit problem.

Same problem. Different technology.

The fix is also the same:
queue the requests, cache the common answers,
throttle individual users so one person can't starve everyone else.

I spent years tuning D365 AOS connection pools
and configuring batch server capacity.

Turns out that was production systems engineering all along.
The same thinking that keeps D365 stable under load
keeps AI applications reliable at scale.

Domain knowledge compounds.
It does not expire.